# 04 Vocabulary Creation

This notebook builds decoder vocabulary artifacts from linearized sequences and target text. Mandatory special tokens are pinned to the first indices.

In [ ]:
from __future__ import annotations
import json
import logging
import re
from pathlib import Path
from typing import Dict, Iterable, List
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')
logger = logging.getLogger('vocab_creation')
ROOT = Path.cwd()
SPECIAL_TOKENS = ['[PAD]', '[UNK]', '[BOS]', '[EOS]', '<sep>']
TOKEN_PATTERN = re.compile(r'\[BOS\]|\[EOS\]|\[PAD\]|\[UNK\]|\[NULL\]|\[EMPTY\]|<sep>|[A-Za-z0-9_@.-]+|[:|,]')

In [ ]:
def tokenize(text: str) -> List[str]:
    return TOKEN_PATTERN.findall(text)

def load_linearized_records() -> List[Dict[str, str]]:
    records = []
    for split in ['train', 'validation', 'test']:
        records.extend(json.loads((ROOT / f'linearized_{split}.json').read_text(encoding='utf-8')))
    return records

def build_vocabulary(records: Iterable[Dict[str, str]]) -> tuple[List[str], Dict[str, int], Dict[str, str]]:
    seen = set(SPECIAL_TOKENS)
    vocab = list(SPECIAL_TOKENS)
    for record in records:
        for field in ['linearized_sequence', 'target_text']:
            for token in tokenize(record.get(field, '')):
                if token not in seen:
                    seen.add(token)
                    vocab.append(token)
    token_to_id = {token: idx for idx, token in enumerate(vocab)}
    id_to_token = {str(idx): token for token, idx in token_to_id.items()}
    return vocab, token_to_id, id_to_token

In [ ]:
records = load_linearized_records()
vocab, token_to_id, id_to_token = build_vocabulary(records)
(ROOT / 'vocab.json').write_text(json.dumps(vocab, indent=2), encoding='utf-8')
(ROOT / 'token_to_id.json').write_text(json.dumps(token_to_id, indent=2), encoding='utf-8')
(ROOT / 'id_to_token.json').write_text(json.dumps(id_to_token, indent=2), encoding='utf-8')
logger.info('Saved vocabulary with %d tokens', len(vocab))

In [ ]:
assert vocab[:5] == SPECIAL_TOKENS
assert token_to_id['[PAD]'] == 0
assert id_to_token[str(token_to_id['<sep>'])] == '<sep>'
summary = pd.DataFrame({'token': vocab, 'id': [token_to_id[token] for token in vocab]})
display(summary.head(25))
display(pd.Series({'num_records': len(records), 'vocab_size': len(vocab)}).to_frame('value'))